In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from PIL import Image
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
from scipy.spatial.transform import Rotation as R
from sklearn.decomposition import PCA
import json

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

In [ ]:
def segment_and_visualize(image_path, device="cuda"):
    """
    Segment image and visualize masks + bounding boxes.
    """
    # Excluded structural elements
    EXCLUDED_LABELS = {
        'wall', 'floor', 'ceiling', 'window', 'door', 'sky', 'ground',
        'wall-other-merged', 'building', 'bottle'
    }
    
    # Load image
    img = Image.open(image_path).convert("RGB")
    img_width, img_height = img.size
    img_np = np.array(img)
    
    # Load model
    model_name = "facebook/mask2former-swin-large-coco-panoptic"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    # Run segmentation
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        panoptic_result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[img.size[::-1]]
        )[0]
    
    segmentation_map = panoptic_result["segmentation"]
    segments_info = panoptic_result["segments_info"]
    
    # Process segments
    segments = []
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        segment_label = model.config.id2label[label_id].lower()
        
        # Skip structural elements
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map == segment_id).cpu()
        mask_np = mask_tensor.numpy()
        
        # Calculate bounding box
        coords = torch.nonzero(mask_tensor)
        if coords.size(0) == 0:
            continue
            
        y_min, x_min = coords.min(dim=0).values
        y_max, x_max = coords.max(dim=0).values
        
        width = (x_max - x_min + 1).item()
        height = (y_max - y_min + 1).item()
        
        bbox_data = {
            'x_min': x_min.item(),
            'y_min': y_min.item(),
            'width': width,
            'height': height,
            'center_x': x_min.item() + width / 2,
            'center_y': y_min.item() + height / 2,
        }
        
        mask_area = mask_tensor.sum().item()
        
        segments.append({
            'id': segment_id,
            'label': model.config.id2label[label_id],
            'bbox': bbox_data,
            'mask_area': mask_area,
            'mask': mask_np
        })
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Original image with bounding boxes
    axes[0].imshow(img_np)
    axes[0].set_title("Bounding Boxes", fontsize=16)
    axes[0].axis('off')
    
    for seg in segments:
        bbox = seg['bbox']
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor='red', facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            f"{seg['label']} (ID: {seg['id']})",
            color='red', fontsize=10, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Segmentation masks (colored)
    axes[1].imshow(img_np)
    axes[1].set_title("Segmentation Masks", fontsize=16)
    axes[1].axis('off')
    
    # Create colored overlay
    # Create colored overlay
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)  # For consistent colors

    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)

    # Ensure both arrays have the same dtype
    img_np_uint8 = img_np.astype(np.uint8)
    overlay = overlay.astype(np.uint8)

    # Blend with original image
    blended = cv2.addWeighted(img_np_uint8, 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    
    plt.tight_layout()
    plt.savefig('segmentation_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return segments, img_width, img_height, img_np

In [ ]:
def estimate_rotation_pca(mask):
    """
    Estimate rotation using PCA on mask pixels.
    Returns rotation in degrees (0-360).
    """
    # Get mask coordinates
    coords = np.argwhere(mask)
    
    if len(coords) < 10:
        return 0.0, 0.0  # Not enough points
    
    # coords are (y, x), convert to (x, y)
    points = coords[:, [1, 0]]
    
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(points)
    
    # First principal component gives the orientation
    principal_axis = pca.components_[0]
    
    # Calculate angle in degrees
    angle_rad = np.arctan2(principal_axis[1], principal_axis[0])
    angle_deg = np.degrees(angle_rad)
    
    # Normalize to 0-360
    angle_deg = angle_deg % 360
    
    # Variance ratio indicates confidence (how elongated is the object?)
    explained_variance_ratio = pca.explained_variance_ratio_[0]
    
    return angle_deg, explained_variance_ratio


def estimate_rotation_min_area_rect(mask):
    """
    Estimate rotation using OpenCV's minimum area rectangle.
    Returns rotation in degrees (0-360).
    """
    # Convert mask to uint8
    mask_uint8 = (mask * 255).astype(np.uint8)
    
    # Find contours
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return 0.0, 0.0
    
    # Get the largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    
    if len(largest_contour) < 5:
        return 0.0, 0.0
    
    # Fit minimum area rectangle
    rect = cv2.minAreaRect(largest_contour)
    angle = rect[2]  # Angle in degrees
    width, height = rect[1]
    
    # OpenCV returns angle in [-90, 0], need to convert to [0, 360]
    # Also need to determine which is the "front" based on aspect ratio
    if width < height:
        angle = angle + 90
    
    # Normalize to 0-360
    angle = angle % 360
    
    # Confidence based on aspect ratio (more elongated = higher confidence)
    aspect_ratio = max(width, height) / (min(width, height) + 1e-6)
    confidence = min(aspect_ratio / 3.0, 1.0)  # Normalize
    
    return angle, confidence


def estimate_rotation_contour_moments(mask):
    """
    Estimate rotation using image moments.
    Returns rotation in degrees (0-360).
    """
    mask_uint8 = (mask * 255).astype(np.uint8)
    
    # Calculate moments
    moments = cv2.moments(mask_uint8)
    
    if moments['m00'] == 0:
        return 0.0, 0.0
    
    # Calculate orientation using central moments
    mu20 = moments['mu20']
    mu02 = moments['mu02']
    mu11 = moments['mu11']
    
    # Calculate angle
    if mu20 == mu02:
        angle_rad = 0
    else:
        angle_rad = 0.5 * np.arctan2(2 * mu11, mu20 - mu02)
    
    angle_deg = np.degrees(angle_rad)
    angle_deg = angle_deg % 360
    
    # Confidence based on how elongated the object is
    elongation = (mu20 - mu02) ** 2 + 4 * mu11 ** 2
    confidence = min(np.sqrt(elongation) / 1000.0, 1.0)
    
    return angle_deg, confidence

In [ ]:
def analyze_rotations(segments, img_np):
    """
    Compute rotations using different methods and visualize.
    """
    results = []
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()
    
    for idx, seg in enumerate(segments[:6]):  # Limit to first 6 for visualization
        mask = seg['mask']
        label = seg['label']
        
        # Compute rotations using all methods
        angle_pca, conf_pca = estimate_rotation_pca(mask)
        angle_mar, conf_mar = estimate_rotation_min_area_rect(mask)
        angle_moments, conf_moments = estimate_rotation_contour_moments(mask)
        
        result = {
            'id': seg['id'],
            'label': label,
            'pca': {'angle': angle_pca, 'confidence': conf_pca},
            'min_area_rect': {'angle': angle_mar, 'confidence': conf_mar},
            'moments': {'angle': angle_moments, 'confidence': conf_moments}
        }
        results.append(result)
        
        # Visualize this object's mask with orientation arrows
        ax = axes[idx]
        
        # Crop to bounding box region
        bbox = seg['bbox']
        x_min, y_min = int(bbox['x_min']), int(bbox['y_min'])
        x_max = int(x_min + bbox['width'])
        y_max = int(y_min + bbox['height'])
        
        # Add padding
        pad = 20
        x_min = max(0, x_min - pad)
        y_min = max(0, y_min - pad)
        x_max = min(img_np.shape[1], x_max + pad)
        y_max = min(img_np.shape[0], y_max + pad)
        
        cropped_img = img_np[y_min:y_max, x_min:x_max]
        cropped_mask = mask[y_min:y_max, x_min:x_max]
        
        # Show image with mask overlay
        overlay = cropped_img.copy()
        overlay[cropped_mask] = [255, 0, 0]  # Red overlay
        blended = cv2.addWeighted(cropped_img, 0.6, overlay, 0.4, 0)
        
        ax.imshow(blended)
        
        # Draw orientation arrows
        center_y, center_x = cropped_mask.shape[0] // 2, cropped_mask.shape[1] // 2
        arrow_length = min(cropped_mask.shape) // 3
        
        # PCA arrow (red)
        angle_rad = np.radians(angle_pca)
        dx = arrow_length * np.cos(angle_rad)
        dy = arrow_length * np.sin(angle_rad)
        ax.arrow(center_x, center_y, dx, dy, color='red', width=3, 
                head_width=15, head_length=10, label='PCA')
        
        # Min Area Rect arrow (blue)
        angle_rad = np.radians(angle_mar)
        dx = arrow_length * np.cos(angle_rad)
        dy = arrow_length * np.sin(angle_rad)
        ax.arrow(center_x, center_y, dx, dy, color='blue', width=2,
                head_width=12, head_length=8, label='Min Area Rect', alpha=0.7)
        
        ax.set_title(f"{label} (ID: {seg['id']})\n"
                    f"PCA: {angle_pca:.1f}° (conf: {conf_pca:.2f})\n"
                    f"MAR: {angle_mar:.1f}° (conf: {conf_mar:.2f})",
                    fontsize=10)
        ax.axis('off')
        ax.legend(loc='upper right', fontsize=8)
    
    # Hide unused subplots
    for idx in range(len(segments), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('rotation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return results

In [ ]:
def print_rotation_results(results):
    """
    Print rotation results in a readable format.
    """
    print("=" * 80)
    print("ROTATION ESTIMATION RESULTS")
    print("=" * 80)
    
    for result in results:
        print(f"\n{result['label']} (ID: {result['id']})")
        print("-" * 40)
        print(f"  PCA Method:")
        print(f"    Angle: {result['pca']['angle']:.1f}°")
        print(f"    Confidence: {result['pca']['confidence']:.3f}")
        print(f"  Minimum Area Rectangle:")
        print(f"    Angle: {result['min_area_rect']['angle']:.1f}°")
        print(f"    Confidence: {result['min_area_rect']['confidence']:.3f}")
        print(f"  Moments Method:")
        print(f"    Angle: {result['moments']['angle']:.1f}°")
        print(f"    Confidence: {result['moments']['confidence']:.3f}")

In [ ]:
# Configuration
IMAGE_PATH = "input/room_test.png"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Analyzing image: {IMAGE_PATH}\n")

# Step 1: Segment and visualize
segments, img_width, img_height, img_np = segment_and_visualize(IMAGE_PATH, DEVICE)

print(f"\nFound {len(segments)} objects")
print("\nObjects detected:")
for seg in segments:
    print(f"  - {seg['label']} (ID: {seg['id']})")

In [ ]:
# Step 2: Analyze rotations
rotation_results = analyze_rotations(segments, img_np)

# Step 3: Print results
print_rotation_results(rotation_results)

In [ ]:
def segment_image_instance(image_path, device="cuda"):
    """
    Use INSTANCE segmentation instead of panoptic.
    """
    EXCLUDED_LABELS = {
        'wall', 'floor', 'ceiling', 'window', 'door', 'sky', 'ground',
        'wall-other-merged', 'building', 'bottle'
    }
    
    img = Image.open(image_path).convert("RGB")
    img_width, img_height = img.size
    img_np = np.array(img)
    
    # Use instance segmentation model
    model_name = "facebook/mask2former-swin-large-coco-instance"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        # Use instance segmentation post-processing
        instance_result = processor.post_process_instance_segmentation(
            outputs, 
            target_sizes=[img.size[::-1]],
            threshold=0.5
        )[0]
    
    print(f"Instance result keys: {instance_result.keys()}")
    
    # The result contains 'segmentation' map and 'segments_info'
    segmentation_map = instance_result["segmentation"]
    segments_info = instance_result["segments_info"]
    
    print(f"Found {len(segments_info)} instances before filtering")
    
    # Process each instance separately
    segments = []
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        score = segment.get('score', 1.0)  # Confidence score
        
        segment_label = model.config.id2label[label_id].lower()
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        # Extract mask for this instance
        mask_tensor = (segmentation_map == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        
        if len(coords) == 0:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        bbox_data = {
            'x_min': int(x_min),
            'y_min': int(y_min),
            'width': int(width),
            'height': int(height),
            'center_x': float(x_min + width / 2),
            'center_y': float(y_min + height / 2),
        }
        
        segments.append({
            'id': segment_id,
            'label': model.config.id2label[label_id],
            'bbox': bbox_data,
            'mask_area': mask_np.sum(),
            'mask': mask_np,
            'confidence': float(score)
        })
    
    print(f"Found {len(segments)} instances after filtering")
    print("\nInstances detected:")
    for seg in segments:
        print(f"  - {seg['label']} (ID: {seg['id']}, conf: {seg['confidence']:.2f})")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(img_np)
    axes[0].set_title("Instance Bounding Boxes", fontsize=16)
    axes[0].axis('off')
    
    for seg in segments:
        bbox = seg['bbox']
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor='red', facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            f"{seg['label']} (ID: {seg['id']}, {seg['confidence']:.2f})",
            color='red', fontsize=8, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Masks
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)
    
    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)
    
    blended = cv2.addWeighted(img_np.astype(np.uint8), 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title("Instance Masks", fontsize=16)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('instance_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return segments, img_width, img_height, img_np

# Test on kitchen image
segments_instance, w, h, img = segment_image_instance(IMAGE_PATH, DEVICE)

In [ ]:
def segment_image_hybrid(image_path, device="cuda"):
    """
    Combine instance and panoptic segmentation for best coverage.
    """
    EXCLUDED_LABELS = {
        'wall', 'floor', 'ceiling', 'window', 'door', 'sky', 'ground',
        'wall-other-merged', 'building'
    }
    
    img = Image.open(image_path).convert("RGB")
    img_width, img_height = img.size
    img_np = np.array(img)
    
    segments = []
    next_id = 0
    
    # === PART 1: Instance Segmentation (for individual objects) ===
    print("Running instance segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-instance"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        instance_result = processor.post_process_instance_segmentation(
            outputs, 
            target_sizes=[img.size[::-1]],
            threshold=0.5
        )[0]
    
    segmentation_map_instance = instance_result["segmentation"]
    segments_info_instance = instance_result["segments_info"]
    
    for segment in segments_info_instance:
        segment_id = segment['id']
        label_id = segment['label_id']
        score = segment.get('score', 1.0)
        
        segment_label = model.config.id2label[label_id].lower()
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map_instance == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        bbox_data = {
            'x_min': int(x_min),
            'y_min': int(y_min),
            'width': int(width),
            'height': int(height),
            'center_x': float(x_min + width / 2),
            'center_y': float(y_min + height / 2),
        }
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': bbox_data,
            'mask_area': mask_np.sum(),
            'mask': mask_np,
            'confidence': float(score),
            'source': 'instance'
        })
        next_id += 1
    
    print(f"  Found {len(segments)} instances")
    
    # === PART 2: Panoptic Segmentation (for structural elements) ===
    print("\nRunning panoptic segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-panoptic"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        panoptic_result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[img.size[::-1]]
        )[0]
    
    segmentation_map_panoptic = panoptic_result["segmentation"]
    segments_info_panoptic = panoptic_result["segments_info"]
    
    # Define "stuff" classes we want to keep from panoptic
    STUFF_CLASSES = {'cabinet', 'light', 'shelf', 'counter', 'countertop'}
    
    stuff_count = 0
    for segment in segments_info_panoptic:
        segment_id = segment['id']
        label_id = segment['label_id']
        
        segment_label = model.config.id2label[label_id].lower()
        
        # Only add "stuff" classes that aren't already in instance results
        if not any(stuff in segment_label for stuff in STUFF_CLASSES):
            continue
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map_panoptic == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        bbox_data = {
            'x_min': int(x_min),
            'y_min': int(y_min),
            'width': int(width),
            'height': int(height),
            'center_x': float(x_min + width / 2),
            'center_y': float(y_min + height / 2),
        }
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': bbox_data,
            'mask_area': mask_np.sum(),
            'mask': mask_np,
            'confidence': 0.9,  # Panoptic doesn't have scores
            'source': 'panoptic'
        })
        next_id += 1
        stuff_count += 1
    
    print(f"  Found {stuff_count} structural elements")
    print(f"\nTotal: {len(segments)} segments")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(img_np)
    axes[0].set_title("Hybrid Segmentation - Bounding Boxes", fontsize=16)
    axes[0].axis('off')
    
    for seg in segments:
        bbox = seg['bbox']
        color = 'blue' if seg['source'] == 'instance' else 'green'
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            f"{seg['label']} ({seg['source'][:4]})",
            color=color, fontsize=8, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Masks
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)
    
    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)
    
    blended = cv2.addWeighted(img_np.astype(np.uint8), 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title("Hybrid Segmentation - Masks", fontsize=16)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('hybrid_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nDetected objects:")
    for seg in segments:
        print(f"  [{seg['source'][:4]}] {seg['label']} (ID: {seg['id']})")
    
    return segments, img_width, img_height, img_np

segments_hybrid, w, h, img = segment_image_hybrid(IMAGE_PATH, DEVICE)

In [ ]:
from scipy import ndimage

def split_merged_masks(segments, min_area_pixels=500):
    """
    Split merged panoptic masks into individual instances using connected components.
    
    Args:
        segments: List of segment dictionaries
        min_area_pixels: Minimum area for a component to be kept as separate instance
        
    Returns:
        Updated segments list with split instances
    """
    new_segments = []
    next_id = max(seg['id'] for seg in segments) + 1
    
    for seg in segments:
        # Only split panoptic "stuff" masks that are likely merged
        if seg['source'] != 'panoptic':
            new_segments.append(seg)
            continue
        
        # Labels that might benefit from splitting
        SPLITTABLE_LABELS = {'light', 'cabinet', 'shelf', 'counter'}
        
        if not any(label in seg['label'].lower() for label in SPLITTABLE_LABELS):
            new_segments.append(seg)
            continue
        
        # Find connected components
        labeled_mask, num_components = ndimage.label(seg['mask'])
        
        if num_components <= 1:
            # Only one component, keep as is
            new_segments.append(seg)
            continue
        
        print(f"\nSplitting {seg['label']} (ID: {seg['id']}) into {num_components} components")
        
        # Create separate segment for each component
        for component_id in range(1, num_components + 1):
            component_mask = (labeled_mask == component_id)
            component_area = component_mask.sum()
            
            # Skip tiny components (noise)
            if component_area < min_area_pixels:
                print(f"  Skipping component {component_id} (too small: {component_area} pixels)")
                continue
            
            # Calculate bounding box for this component
            coords = np.argwhere(component_mask)
            y_min, x_min = coords.min(axis=0)
            y_max, x_max = coords.max(axis=0)
            width = x_max - x_min + 1
            height = y_max - y_min + 1
            
            bbox_data = {
                'x_min': int(x_min),
                'y_min': int(y_min),
                'width': int(width),
                'height': int(height),
                'center_x': float(x_min + width / 2),
                'center_y': float(y_min + height / 2),
            }
            
            new_segments.append({
                'id': next_id,
                'label': seg['label'],  # Keep same label
                'bbox': bbox_data,
                'mask_area': int(component_area),
                'mask': component_mask,
                'confidence': seg['confidence'],
                'source': 'panoptic_split'
            })
            
            print(f"  Component {component_id}: area={component_area} pixels, bbox={bbox_data['width']}x{bbox_data['height']}")
            next_id += 1
    
    print(f"\nTotal segments: {len(segments)} → {len(new_segments)}")
    return new_segments


def visualize_split_segments(segments, img_np):
    """
    Visualize the split segments.
    """
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Bounding boxes
    axes[0].imshow(img_np)
    axes[0].set_title("Split Segmentation - Bounding Boxes", fontsize=16)
    axes[0].axis('off')
    
    # Color by source
    color_map = {
        'instance': 'blue',
        'panoptic': 'green',
        'panoptic_split': 'red'
    }
    
    for seg in segments:
        bbox = seg['bbox']
        color = color_map.get(seg['source'], 'gray')
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            f"{seg['label']} (ID: {seg['id']})",
            color=color, fontsize=8, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Masks
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)
    
    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)
    
    blended = cv2.addWeighted(img_np.astype(np.uint8), 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title("Split Segmentation - Masks", fontsize=16)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('split_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary
    print("\n" + "="*80)
    print("SPLIT SEGMENTATION SUMMARY")
    print("="*80)
    
    by_source = {}
    for seg in segments:
        source = seg['source']
        by_source[source] = by_source.get(source, 0) + 1
    
    print(f"\nSegments by source:")
    for source, count in by_source.items():
        print(f"  {source}: {count}")
    
    print(f"\nAll segments:")
    for seg in segments:
        source_marker = {
            'instance': '[inst]',
            'panoptic': '[pano]',
            'panoptic_split': '[SPLIT]'
        }
        marker = source_marker.get(seg['source'], '[?]')
        print(f"  {marker} {seg['label']} (ID: {seg['id']})")

In [ ]:
# Split the merged masks
segments_split = split_merged_masks(segments_hybrid, min_area_pixels=500)

# Visualize
visualize_split_segments(segments_split, img)

In [ ]:
def resolve_bbox_overlaps(segments, img_width, img_height):
    """
    Resolve bounding box overlaps when masks don't actually overlap.
    
    Strategy:
    1. Find overlapping bounding boxes
    2. Check if masks actually overlap
    3. If boxes overlap but masks don't, shrink the larger box
    4. Only apply to structural elements (panoptic), not instances (chairs can overlap)
    """
    
    def bbox_overlap_area(bbox1, bbox2):
        """Calculate overlapping area between two bounding boxes."""
        x1_min, y1_min = bbox1['x_min'], bbox1['y_min']
        x1_max = x1_min + bbox1['width']
        y1_max = y1_min + bbox1['height']
        
        x2_min, y2_min = bbox2['x_min'], bbox2['y_min']
        x2_max = x2_min + bbox2['width']
        y2_max = y2_min + bbox2['height']
        
        # Calculate intersection
        x_overlap = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
        y_overlap = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
        
        overlap_area = x_overlap * y_overlap
        
        # Calculate IoU (Intersection over Union)
        bbox1_area = bbox1['width'] * bbox1['height']
        bbox2_area = bbox2['width'] * bbox2['height']
        union_area = bbox1_area + bbox2_area - overlap_area
        
        iou = overlap_area / union_area if union_area > 0 else 0
        
        return overlap_area, iou
    
    def masks_actually_overlap(mask1, mask2):
        """Check if two masks have any overlapping pixels."""
        overlap = np.logical_and(mask1, mask2)
        return overlap.sum() > 0
    
    def compute_tight_bbox(mask, exclude_mask=None):
        """
        Compute tighter bounding box by excluding another mask's region.
        """
        if exclude_mask is not None:
            # Remove the excluded region from the mask
            adjusted_mask = np.logical_and(mask, ~exclude_mask)
        else:
            adjusted_mask = mask
        
        coords = np.argwhere(adjusted_mask)
        
        if len(coords) == 0:
            return None
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        return {
            'x_min': int(x_min),
            'y_min': int(y_min),
            'width': int(width),
            'height': int(height),
            'center_x': float(x_min + width / 2),
            'center_y': float(y_min + height / 2),
        }
    
    # Identify structural elements (panoptic/split) vs instances
    structural_elements = [s for s in segments if s['source'] in ['panoptic', 'panoptic_split']]
    instance_elements = [s for s in segments if s['source'] == 'instance']
    
    print("\n" + "="*80)
    print("RESOLVING BOUNDING BOX OVERLAPS")
    print("="*80)
    
    resolved_segments = segments.copy()
    resolution_count = 0
    
    # Only resolve overlaps between structural elements
    # (instances like chairs can overlap - they're at different depths)
    for i, seg1 in enumerate(structural_elements):
        for j, seg2 in enumerate(structural_elements):
            if i >= j:  # Avoid duplicate comparisons
                continue
            
            # Check bounding box overlap
            overlap_area, iou = bbox_overlap_area(seg1['bbox'], seg2['bbox'])
            
            if overlap_area == 0:
                continue
            
            # Significant overlap in bounding boxes
            if iou > 0.1:  # 10% IoU threshold
                # Check if masks actually overlap
                masks_overlap = masks_actually_overlap(seg1['mask'], seg2['mask'])
                
                if not masks_overlap:
                    print(f"\nBBox overlap without mask overlap:")
                    print(f"  {seg1['label']} (ID: {seg1['id']}) vs {seg2['label']} (ID: {seg2['id']})")
                    print(f"  BBox IoU: {iou:.3f}, Mask overlap: {masks_overlap}")
                    
                    # Determine which is larger
                    area1 = seg1['bbox']['width'] * seg1['bbox']['height']
                    area2 = seg2['bbox']['width'] * seg2['bbox']['height']
                    
                    if area1 > area2:
                        larger_seg = seg1
                        smaller_seg = seg2
                    else:
                        larger_seg = seg2
                        smaller_seg = seg1
                    
                    # Compute tighter bbox for larger segment excluding smaller
                    new_bbox = compute_tight_bbox(larger_seg['mask'], smaller_seg['mask'])
                    
                    if new_bbox is not None:
                        # Find and update the segment in resolved_segments
                        for idx, seg in enumerate(resolved_segments):
                            if seg['id'] == larger_seg['id']:
                                old_bbox = seg['bbox']
                                resolved_segments[idx]['bbox'] = new_bbox
                                resolved_segments[idx]['bbox_adjusted'] = True
                                
                                print(f"  Adjusted {larger_seg['label']} (ID: {larger_seg['id']}):")
                                print(f"    Old bbox: {old_bbox['width']}x{old_bbox['height']}")
                                print(f"    New bbox: {new_bbox['width']}x{new_bbox['height']}")
                                
                                resolution_count += 1
                                break
    
    print(f"\nResolved {resolution_count} bounding box overlaps")
    
    return resolved_segments


def visualize_resolved_segments(segments, img_np):
    """
    Visualize segments with resolved bounding boxes.
    Highlight adjusted boxes in orange.
    """
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Bounding boxes
    axes[0].imshow(img_np)
    axes[0].set_title("Resolved Bounding Boxes", fontsize=16)
    axes[0].axis('off')
    
    for seg in segments:
        bbox = seg['bbox']
        
        # Color based on whether bbox was adjusted
        if seg.get('bbox_adjusted', False):
            color = 'orange'
            linewidth = 3
        else:
            color_map = {
                'instance': 'blue',
                'panoptic': 'green',
                'panoptic_split': 'red'
            }
            color = color_map.get(seg['source'], 'gray')
            linewidth = 2
        
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=linewidth, edgecolor=color, facecolor='none'
        )
        axes[0].add_patch(rect)
        
        label_text = f"{seg['label']} (ID: {seg['id']})"
        if seg.get('bbox_adjusted', False):
            label_text += " [ADJUSTED]"
        
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            label_text,
            color=color, fontsize=8, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Masks (unchanged)
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)
    
    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)
    
    blended = cv2.addWeighted(img_np.astype(np.uint8), 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title("Masks (unchanged)", fontsize=16)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('resolved_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Resolve overlaps
segments_resolved = resolve_bbox_overlaps(segments_split, img.shape[1], img.shape[0])

# Visualize
visualize_resolved_segments(segments_resolved, img)

In [ ]:
def aggressive_spatial_split(segments, img_width, img_height):
    """
    Apply spatial heuristics to split complex masks that connected components missed.
    
    Strategies:
    1. Split L-shaped cabinets by finding the "corner"
    2. Split horizontally-arranged lights by vertical cuts
    3. Split vertically-arranged shelves by horizontal cuts
    """
    
    def detect_l_shape(mask, bbox):
        """
        Detect if a mask is L-shaped by checking if it has two dominant regions.
        """
        # Sample the mask at different heights to see occupancy
        height = bbox['height']
        width = bbox['width']
        
        # Check top 1/3, middle 1/3, bottom 1/3
        y_min = bbox['y_min']
        x_min = bbox['x_min']
        
        top_region = mask[y_min:y_min + height//3, x_min:x_min + width]
        mid_region = mask[y_min + height//3:y_min + 2*height//3, x_min:x_min + width]
        bot_region = mask[y_min + 2*height//3:y_min + height, x_min:x_min + width]
        
        top_density = top_region.sum() / (top_region.size + 1e-6)
        mid_density = mid_region.sum() / (mid_region.size + 1e-6)
        bot_density = bot_region.sum() / (bot_region.size + 1e-6)
        
        # L-shape: high density in one vertical region, mixed in others
        # Or check if left vs right occupancy differs significantly
        
        # Check left vs right occupancy
        left_half = mask[y_min:y_min + height, x_min:x_min + width//2]
        right_half = mask[y_min:y_min + height, x_min + width//2:x_min + width]
        
        left_density = left_half.sum() / (left_half.size + 1e-6)
        right_density = right_half.sum() / (right_half.size + 1e-6)
        
        # L-shaped if one side is much denser
        density_ratio = max(left_density, right_density) / (min(left_density, right_density) + 0.1)
        
        return density_ratio > 2.0, density_ratio
    
    def split_horizontally_arranged_objects(mask, bbox, num_expected=3):
        """
        Split objects arranged horizontally (like pendant lights).
        """
        # Divide mask into vertical strips
        x_min = bbox['x_min']
        width = bbox['width']
        
        strip_width = width // num_expected
        submasks = []
        
        for i in range(num_expected):
            x_start = x_min + i * strip_width
            x_end = x_min + (i + 1) * strip_width if i < num_expected - 1 else x_min + width
            
            submask = np.zeros_like(mask)
            submask[:, x_start:x_end] = mask[:, x_start:x_end]
            
            if submask.sum() > 500:  # Minimum area
                submasks.append(submask)
        
        return submasks
    
    def split_l_shaped_cabinet(mask, bbox):
        """
        Split L-shaped cabinet by finding the corner point.
        """
        # Find the horizontal and vertical "spines"
        row_sums = mask.sum(axis=1)
        col_sums = mask.sum(axis=0)
        
        # Find where the horizontal spine ends (likely the corner)
        # The bottom section should have consistent width, top section should be narrower
        
        # Try splitting at different heights and pick the best
        best_split_y = None
        best_score = 0
        
        for split_y in range(bbox['y_min'] + bbox['height']//4, 
                            bbox['y_min'] + 3*bbox['height']//4, 
                            10):
            top_mask = mask.copy()
            bot_mask = mask.copy()
            
            top_mask[split_y:, :] = False
            bot_mask[:split_y, :] = False
            
            # Score: both parts should have reasonable area
            top_area = top_mask.sum()
            bot_area = bot_mask.sum()
            
            if top_area > 5000 and bot_area > 5000:
                # Prefer splits that separate into more rectangular regions
                score = min(top_area, bot_area)
                if score > best_score:
                    best_score = score
                    best_split_y = split_y
        
        if best_split_y is not None:
            top_mask = mask.copy()
            bot_mask = mask.copy()
            top_mask[best_split_y:, :] = False
            bot_mask[:best_split_y, :] = False
            return [top_mask, bot_mask]
        
        return [mask]
    
    new_segments = []
    next_id = max(seg['id'] for seg in segments) + 1
    
    print("\n" + "="*80)
    print("AGGRESSIVE SPATIAL SPLITTING")
    print("="*80)
    
    for seg in segments:
        label_lower = seg['label'].lower()
        
        # Strategy 1: Split L-shaped cabinets
        if 'cabinet' in label_lower and seg['source'] in ['panoptic', 'panoptic_split']:
            is_l_shape, ratio = detect_l_shape(seg['mask'], seg['bbox'])
            
            if is_l_shape:
                print(f"\nDetected L-shaped {seg['label']} (ID: {seg['id']}, ratio: {ratio:.2f})")
                submasks = split_l_shaped_cabinet(seg['mask'], seg['bbox'])
                
                if len(submasks) > 1:
                    print(f"  Split into {len(submasks)} parts")
                    for i, submask in enumerate(submasks):
                        coords = np.argwhere(submask)
                        if len(coords) == 0:
                            continue
                        
                        y_min, x_min = coords.min(axis=0)
                        y_max, x_max = coords.max(axis=0)
                        width = x_max - x_min + 1
                        height = y_max - y_min + 1
                        
                        new_segments.append({
                            'id': next_id,
                            'label': seg['label'],
                            'bbox': {
                                'x_min': int(x_min),
                                'y_min': int(y_min),
                                'width': int(width),
                                'height': int(height),
                                'center_x': float(x_min + width / 2),
                                'center_y': float(y_min + height / 2),
                            },
                            'mask_area': int(submask.sum()),
                            'mask': submask,
                            'confidence': seg['confidence'],
                            'source': 'spatial_split'
                        })
                        next_id += 1
                    continue
        
        # Strategy 2: Split horizontally-arranged lights
        if 'light' in label_lower and seg['bbox']['width'] > seg['bbox']['height'] * 1.5:
            print(f"\nDetected horizontal light arrangement {seg['label']} (ID: {seg['id']})")
            submasks = split_horizontally_arranged_objects(seg['mask'], seg['bbox'], num_expected=3)
            
            if len(submasks) > 1:
                print(f"  Split into {len(submasks)} lights")
                for i, submask in enumerate(submasks):
                    coords = np.argwhere(submask)
                    if len(coords) == 0:
                        continue
                    
                    y_min, x_min = coords.min(axis=0)
                    y_max, x_max = coords.max(axis=0)
                    width = x_max - x_min + 1
                    height = y_max - y_min + 1
                    
                    new_segments.append({
                        'id': next_id,
                        'label': seg['label'],
                        'bbox': {
                            'x_min': int(x_min),
                            'y_min': int(y_min),
                            'width': int(width),
                            'height': int(height),
                            'center_x': float(x_min + width / 2),
                            'center_y': float(y_min + height / 2),
                        },
                        'mask_area': int(submask.sum()),
                        'mask': submask,
                        'confidence': seg['confidence'],
                        'source': 'spatial_split'
                    })
                    next_id += 1
                continue
        
        # Keep unsplit
        new_segments.append(seg)
    
    print(f"\nSegments: {len(segments)} → {len(new_segments)}")
    return new_segments

In [ ]:
# Apply aggressive spatial splitting
segments_final = aggressive_spatial_split(segments_resolved, img.shape[1], img.shape[0])

# Visualize
visualize_resolved_segments(segments_final, img)

# Print summary
print("\n" + "="*80)
print("FINAL SEGMENTATION")
print("="*80)
for seg in segments_final:
    source_marker = {
        'instance': '[inst]',
        'panoptic': '[pano]',
        'panoptic_split': '[SPLIT]',
        'spatial_split': '[SPATIAL]'
    }
    marker = source_marker.get(seg['source'], '[?]')
    print(f"  {marker} {seg['label']} (ID: {seg['id']})")

In [ ]:
def add_rotation_estimates(segments):
    """
    Add MAR rotation estimates to all segments.
    """
    from cv2 import minAreaRect, findContours, RETR_EXTERNAL, CHAIN_APPROX_SIMPLE
    
    print("\n" + "="*80)
    print("COMPUTING ROTATIONS")
    print("="*80)
    
    for seg in segments:
        mask = seg['mask']
        mask_uint8 = (mask * 255).astype(np.uint8)
        
        contours, _ = findContours(mask_uint8, RETR_EXTERNAL, CHAIN_APPROX_SIMPLE)
        
        if not contours:
            seg['rotation_deg'] = 0.0
            seg['rotation_confidence'] = 0.0
            continue
        
        largest_contour = max(contours, key=lambda c: len(c))
        
        if len(largest_contour) < 5:
            seg['rotation_deg'] = 0.0
            seg['rotation_confidence'] = 0.0
            continue
        
        rect = minAreaRect(largest_contour)
        angle = rect[2]
        width, height = rect[1]
        
        # Adjust angle based on aspect ratio
        if width < height:
            angle = angle + 90
        
        angle = angle % 360
        
        # Confidence based on elongation
        aspect_ratio = max(width, height) / (min(width, height) + 1e-6)
        confidence = min(aspect_ratio / 3.0, 1.0)
        
        seg['rotation_deg'] = float(angle)
        seg['rotation_confidence'] = float(confidence)
        
        # Only print for objects with meaningful rotation
        if confidence > 0.3:
            print(f"  {seg['label']} (ID: {seg['id']}): {angle:.1f}° (conf: {confidence:.2f})")
    
    return segments


def visualize_with_rotations(segments, img_np):
    """
    Visualize segments with rotation arrows.
    """
    fig, ax = plt.subplots(1, 1, figsize=(20, 12))
    ax.imshow(img_np)
    ax.set_title("Final Segmentation with Rotations", fontsize=16)
    ax.axis('off')
    
    color_map = {
        'instance': 'blue',
        'panoptic': 'green',
        'panoptic_split': 'red',
        'spatial_split': 'orange'
    }
    
    for seg in segments:
        bbox = seg['bbox']
        color = color_map.get(seg['source'], 'gray')
        
        # Draw bounding box
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw label
        label_text = f"{seg['label']} (ID: {seg['id']})"
        if seg.get('rotation_confidence', 0) > 0.3:
            label_text += f"\n{seg['rotation_deg']:.0f}°"
        
        ax.text(
            bbox['x_min'], bbox['y_min'] - 5,
            label_text,
            color=color, fontsize=8, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
        
        # Draw rotation arrow for objects with meaningful rotation
        if seg.get('rotation_confidence', 0) > 0.3:
            center_x = bbox['center_x']
            center_y = bbox['center_y']
            arrow_length = min(bbox['width'], bbox['height']) / 3
            
            angle_rad = np.radians(seg['rotation_deg'])
            dx = arrow_length * np.cos(angle_rad)
            dy = arrow_length * np.sin(angle_rad)
            
            ax.arrow(center_x, center_y, dx, dy, 
                    color=color, width=2, head_width=10, head_length=8, alpha=0.7)
    
    plt.tight_layout()
    plt.savefig('final_segmentation_with_rotations.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Add rotations to segments
segments_with_rotations = add_rotation_estimates(segments_final)

# Visualize with rotation arrows
visualize_with_rotations(segments_with_rotations, img)

# Print final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"Total objects: {len(segments_with_rotations)}")
print(f"  Instance objects: {sum(1 for s in segments_with_rotations if s['source'] == 'instance')}")
print(f"  Structural elements: {sum(1 for s in segments_with_rotations if s['source'] != 'instance')}")
print(f"  Objects with rotation: {sum(1 for s in segments_with_rotations if s.get('rotation_confidence', 0) > 0.3)}")

In [ ]:
### PCA Focused ###

# Relative Rotation with Horizontal Projection
# Key Ideas:

# Use PCA but extract the horizontal component for orientation
# Pick a reference object (e.g., largest furniture piece) and set it to 0°
# Compute all other rotations relative to the reference
# For vertical objects (like refrigerators), use the principal component that's most horizontal

In [ ]:
def estimate_rotation_pca_with_components(mask):
    """
    Estimate rotation using PCA, returning both principal components.
    Returns rotation angle and both principal component vectors.
    """
    coords = np.argwhere(mask)
    
    if len(coords) < 10:
        return 0.0, 0.0, None, None
    
    # coords are (y, x), convert to (x, y)
    points = coords[:, [1, 0]]
    
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(points)
    
    # Get both principal components
    pc1 = pca.components_[0]  # First (dominant) principal component
    pc2 = pca.components_[1]  # Second principal component
    
    # Calculate angle from first PC
    angle_rad = np.arctan2(pc1[1], pc1[0])
    angle_deg = np.degrees(angle_rad) % 360
    
    # Variance ratio indicates confidence
    explained_variance_ratio = pca.explained_variance_ratio_[0]
    
    return angle_deg, explained_variance_ratio, pc1, pc2


def get_horizontal_orientation_angle(mask, bbox):
    """
    Get the horizontal orientation angle, handling vertical objects.
    
    For vertical objects (height >> width), uses the secondary component.
    For horizontal objects, uses the primary component.
    
    Returns: (angle_deg, confidence, is_vertical)
    """
    angle_deg, confidence, pc1, pc2 = estimate_rotation_pca_with_components(mask)
    
    if pc1 is None:
        return 0.0, 0.0, False
    
    # Check if object is primarily vertical in the image
    aspect_ratio = bbox['height'] / (bbox['width'] + 1e-6)
    
    # Check if PC1 is too vertical (more Y component than X)
    pc1_horizontal_extent = abs(pc1[0])
    pc1_vertical_extent = abs(pc1[1])
    pc1_is_vertical = pc1_vertical_extent > pc1_horizontal_extent * 1.5
    
    # For tall objects with vertical PC1, use PC2 for horizontal orientation
    if aspect_ratio > 2.0 and pc1_is_vertical and pc2 is not None:
        # Use second principal component for horizontal orientation
        angle_rad = np.arctan2(pc2[1], pc2[0])
        angle_deg = np.degrees(angle_rad) % 360
        is_vertical = True
        print(f"  Object is vertical (aspect: {aspect_ratio:.2f}), using PC2 for horizontal orientation")
    else:
        # Use first principal component
        is_vertical = False
    
    return angle_deg, confidence, is_vertical


def estimate_relative_rotations(segments, reference_id=None):
    """
    Estimate rotations relative to a reference object.
    
    Args:
        segments: List of segment dictionaries
        reference_id: ID of reference object (if None, picks automatically)
        
    Returns:
        List of rotation results with relative angles
    """
    print("\n" + "="*80)
    print("COMPUTING RELATIVE ROTATIONS")
    print("="*80)
    
    # Step 1: Compute horizontal orientation angles for all objects
    rotation_results = []
    
    for seg in segments:
        angle_deg, confidence, is_vertical = get_horizontal_orientation_angle(
            seg['mask'], 
            seg['bbox']
        )
        
        rotation_results.append({
            'id': seg['id'],
            'label': seg['label'],
            'absolute_angle': angle_deg,
            'confidence': confidence,
            'is_vertical': is_vertical,
            'bbox_area': seg['bbox']['width'] * seg['bbox']['height']
        })
    
    # Step 2: Pick reference object (if not specified)
    if reference_id is None:
        # Pick largest object with reasonable confidence as reference
        candidates = [r for r in rotation_results if r['confidence'] > 0.3]
        
        if candidates:
            # Use largest object as reference
            reference = max(candidates, key=lambda x: x['bbox_area'])
            reference_id = reference['id']
        else:
            # Fallback: just use first object
            reference = rotation_results[0]
            reference_id = reference['id']
    else:
        reference = next(r for r in rotation_results if r['id'] == reference_id)
    
    reference_angle = reference['absolute_angle']
    
    print(f"\nReference object: {reference['label']} (ID: {reference_id})")
    print(f"  Reference angle: {reference_angle:.1f}°")
    print(f"  Setting reference to 0°\n")
    
    # Step 3: Compute relative angles
    for result in rotation_results:
        # Compute angle relative to reference
        relative_angle = (result['absolute_angle'] - reference_angle) % 360
        
        # Normalize to [-180, 180] for easier interpretation
        if relative_angle > 180:
            relative_angle -= 360
        
        result['relative_angle'] = relative_angle
        result['is_reference'] = (result['id'] == reference_id)
        
        if result['is_reference']:
            print(f"✓ {result['label']} (ID: {result['id']}): 0° [REFERENCE]")
        else:
            print(f"  {result['label']} (ID: {result['id']}): {relative_angle:.1f}° "
                  f"(conf: {result['confidence']:.2f})")
    
    return rotation_results, reference_id


def visualize_relative_rotations(segments, rotation_results, img_np, reference_id):
    """
    Visualize relative rotations with arrows showing orientation relative to reference.
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()
    
    # Get reference angle for visualization
    reference = next(r for r in rotation_results if r['id'] == reference_id)
    reference_angle = reference['absolute_angle']
    
    for idx, seg in enumerate(segments[:6]):
        if idx >= 6:
            break
            
        ax = axes[idx]
        mask = seg['mask']
        bbox = seg['bbox']
        
        # Find rotation result for this segment
        rot_result = next(r for r in rotation_results if r['id'] == seg['id'])
        
        # Crop to bounding box region
        x_min, y_min = int(bbox['x_min']), int(bbox['y_min'])
        x_max = int(x_min + bbox['width'])
        y_max = int(y_min + bbox['height'])
        
        pad = 20
        x_min = max(0, x_min - pad)
        y_min = max(0, y_min - pad)
        x_max = min(img_np.shape[1], x_max + pad)
        y_max = min(img_np.shape[0], y_max + pad)
        
        cropped_img = img_np[y_min:y_max, x_min:x_max]
        cropped_mask = mask[y_min:y_max, x_min:x_max]
        
        # Show image with mask overlay
        overlay = cropped_img.copy()
        overlay[cropped_mask] = [255, 0, 0]
        blended = cv2.addWeighted(cropped_img, 0.6, overlay, 0.4, 0)
        
        ax.imshow(blended)
        
        # Draw orientation arrow
        center_y, center_x = cropped_mask.shape[0] // 2, cropped_mask.shape[1] // 2
        arrow_length = min(cropped_mask.shape) // 3
        
        # Arrow showing absolute angle
        angle_rad = np.radians(rot_result['absolute_angle'])
        dx = arrow_length * np.cos(angle_rad)
        dy = arrow_length * np.sin(angle_rad)
        
        color = 'green' if rot_result['is_reference'] else 'red'
        ax.arrow(center_x, center_y, dx, dy, 
                color=color, width=3, head_width=15, head_length=10)
        
        # Title
        title = f"{seg['label']} (ID: {seg['id']})\n"
        if rot_result['is_reference']:
            title += "REFERENCE: 0°"
        else:
            title += f"Relative: {rot_result['relative_angle']:.1f}°"
        title += f"\n(conf: {rot_result['confidence']:.2f})"
        
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(len(segments), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('relative_rotation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Compute relative rotations
rotation_results, reference_id = estimate_relative_rotations(segments_final)

# Visualize
visualize_relative_rotations(segments_final, rotation_results, img_np, reference_id)

# Print spatial relationships
print("\n" + "="*80)
print("SPATIAL RELATIONSHIPS")
print("="*80)

for i, r1 in enumerate(rotation_results):
    if r1['is_reference']:
        continue
    
    angle_diff = abs(r1['relative_angle'])
    
    # Normalize to [0, 90] for relationship detection
    if angle_diff > 90:
        angle_diff = 180 - angle_diff
    
    if angle_diff < 15:
        relationship = "PARALLEL"
    elif 75 < angle_diff < 105:
        relationship = "PERPENDICULAR"
    else:
        relationship = f"ANGLED ({r1['relative_angle']:.1f}°)"
    
    print(f"{r1['label']} is {relationship} to reference")

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
from sklearn.decomposition import PCA

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

In [ ]:
def segment_image_simple(image_path, device="cuda"):
    """
    Simple hybrid segmentation: instance + panoptic, no splitting.
    """
    EXCLUDED_LABELS = {
        'wall', 'floor', 'ceiling', 'window', 'door', 'sky', 'ground',
        'wall-other-merged', 'building'
    }
    
    EXCLUDED_SMALL_OBJECTS = {

    }
    
    img = Image.open(image_path).convert("RGB")
    img_width, img_height = img.size
    img_np = np.array(img)
    
    segments = []
    next_id = 0
    
    # === Instance Segmentation ===
    print("Running instance segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-instance"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        instance_result = processor.post_process_instance_segmentation(
            outputs, 
            target_sizes=[img.size[::-1]],
            threshold=0.45
        )[0]
    
    segmentation_map = instance_result["segmentation"]
    segments_info = instance_result["segments_info"]

    # DEBUG: Print all detections before filtering
    print("\n--- ALL DETECTIONS (before filtering) ---")
    for segment in segments_info:
        label_id = segment['label_id']
        score = segment.get('score', 1.0)
        segment_label = model.config.id2label[label_id]
        print(f"  {segment_label}: confidence={score:.3f}")
    print("--- End detections ---\n")
    
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        score = segment.get('score', 1.0)
        segment_label = model.config.id2label[label_id].lower()
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        if segment_label in EXCLUDED_SMALL_OBJECTS:
            continue
        
        mask_tensor = (segmentation_map == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0:
            continue
        
        # Skip very small objects
        if mask_np.sum() < 2000:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': {
                'x_min': int(x_min),
                'y_min': int(y_min),
                'width': int(width),
                'height': int(height),
                'center_x': float(x_min + width / 2),
                'center_y': float(y_min + height / 2),
            },
            'mask': mask_np,
            'confidence': float(score),
            'source': 'instance'
        })
        next_id += 1
    
    print(f"  Found {len(segments)} instances")
    
    # === Panoptic Segmentation ===
    print("Running panoptic segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-panoptic"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        panoptic_result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[img.size[::-1]]
        )[0]
    
    segmentation_map = panoptic_result["segmentation"]
    segments_info = panoptic_result["segments_info"]

    print("\n--- PANOPTIC DETECTIONS ---")
    for segment in segments_info:
        label_id = segment['label_id']
        segment_label = model.config.id2label[label_id]
        print(f"  {segment_label}")
    print("--- End panoptic ---\n")
    
    STUFF_CLASSES = {
        'cabinet', 'light', 'shelf', 'counter', 'countertop',
        'table', 'desk', 'dining table', 'coffee table'
    }
    
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        segment_label = model.config.id2label[label_id].lower()
        
        if not any(stuff in segment_label for stuff in STUFF_CLASSES):
            continue
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': {
                'x_min': int(x_min),
                'y_min': int(y_min),
                'width': int(width),
                'height': int(height),
                'center_x': float(x_min + width / 2),
                'center_y': float(y_min + height / 2),
            },
            'mask': mask_np,
            'confidence': 0.9,
            'source': 'panoptic'
        })
        next_id += 1
    
    print(f"  Found {len(segments)} total objects")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Bounding boxes
    axes[0].imshow(img_np)
    axes[0].set_title("Segmentation Results", fontsize=16)
    axes[0].axis('off')
    
    for seg in segments:
        bbox = seg['bbox']
        color = 'blue' if seg['source'] == 'instance' else 'green'
        rect = Rectangle(
            (bbox['x_min'], bbox['y_min']),
            bbox['width'], bbox['height'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            bbox['x_min'], bbox['y_min'] - 5,
            f"{seg['label']} (ID: {seg['id']})",
            color=color, fontsize=10, weight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Masks
    overlay = np.zeros_like(img_np, dtype=np.uint8)
    np.random.seed(42)
    
    for seg in segments:
        color = np.random.randint(0, 255, 3, dtype=np.uint8)
        mask_3d = np.stack([seg['mask']] * 3, axis=-1)
        overlay = np.where(mask_3d, color, overlay)
    
    blended = cv2.addWeighted(img_np.astype(np.uint8), 0.6, overlay, 0.4, 0)
    axes[1].imshow(blended)
    axes[1].set_title("Masks", fontsize=16)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('simple_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return segments, img_width, img_height, img_np

In [ ]:
def estimate_relative_rotations_correct(segments, reference_id=None):
    """
    Compute angles BETWEEN vectors using dot product.
    """
    print("\n" + "="*80)
    print("RELATIVE ROTATION ESTIMATION (DOT PRODUCT)")
    print("="*80)
    
    # Get PCA vectors for all objects
    pca_results = []
    
    for seg in segments:
        coords = np.argwhere(seg['mask'])
        
        if len(coords) < 10:
            pca_results.append({
                'id': seg['id'],
                'label': seg['label'],
                'pc1': None,
                'confidence': 0.0,
                'bbox_area': seg['bbox']['width'] * seg['bbox']['height']
            })
            continue
        
        points = coords[:, [1, 0]]
        pca = PCA(n_components=2)
        pca.fit(points)
        
        pc1 = pca.components_[0]
        confidence = pca.explained_variance_ratio_[0]
        
        pca_results.append({
            'id': seg['id'],
            'label': seg['label'],
            'pc1': pc1,
            'confidence': confidence,
            'bbox_area': seg['bbox']['width'] * seg['bbox']['height']
        })
    
    # Pick reference
    if reference_id is None:
        candidates = [r for r in pca_results if r['pc1'] is not None and r['confidence'] > 0.3]
        reference = max(candidates, key=lambda x: x['bbox_area']) if candidates else pca_results[0]
        reference_id = reference['id']
    else:
        reference = next(r for r in pca_results if r['id'] == reference_id)
    
    print(f"\nReference: {reference['label']} (ID: {reference_id})")
    print(f"  PC1: [{reference['pc1'][0]:.3f}, {reference['pc1'][1]:.3f}]\n")
    
    # Compute angles between each vector and reference
    rotation_results = []
    
    for result in pca_results:
        if result['pc1'] is None:
            rotation_results.append({
                **result,
                'angle_to_reference': 0.0,
                'is_reference': False
            })
            continue
        
        is_reference = (result['id'] == reference_id)
        
        if is_reference:
            angle_between = 0.0
        else:
            print(f"Computing angle for {result['label']} with pc: {result['pc1']}")
            # Dot product between vectors
            dot_product = np.dot(result['pc1'], reference['pc1'])
            
            # Clamp to avoid numerical errors in arccos
            dot_product = np.clip(dot_product, -1.0, 1.0)
            
            # Angle between vectors (using abs to get magnitude)
            angle_between = np.degrees(np.arccos(np.abs(dot_product)))
        
        rotation_results.append({
            **result,
            'angle_to_reference': angle_between,
            'is_reference': is_reference
        })
        
        if is_reference:
            print(f"✓ {result['label']} (ID: {result['id']}): 0° [REFERENCE]")
        else:
            print(f"  {result['label']} (ID: {result['id']}): {angle_between:.1f}° "
                  f"(conf: {result['confidence']:.2f})")
    
    return rotation_results, reference_id


def visualize_relative_rotations(segments, rotation_results, img_np, reference_id):
    """Visualize with colored arrows."""
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()
    
    for idx, seg in enumerate(segments[:6]):
        if idx >= 6:
            break
            
        ax = axes[idx]
        rot_result = next(r for r in rotation_results if r['id'] == seg['id'])
        
        bbox = seg['bbox']
        x_min, y_min = int(bbox['x_min']), int(bbox['y_min'])
        x_max = int(x_min + bbox['width'])
        y_max = int(y_min + bbox['height'])
        
        pad = 20
        x_min = max(0, x_min - pad)
        y_min = max(0, y_min - pad)
        x_max = min(img_np.shape[1], x_max + pad)
        y_max = min(img_np.shape[0], y_max + pad)
        
        cropped_img = img_np[y_min:y_max, x_min:x_max]
        cropped_mask = seg['mask'][y_min:y_max, x_min:x_max]
        
        overlay = cropped_img.copy()
        overlay[cropped_mask] = [255, 0, 0]
        blended = cv2.addWeighted(cropped_img, 0.6, overlay, 0.4, 0)
        
        ax.imshow(blended)
        
        center_y, center_x = cropped_mask.shape[0] // 2, cropped_mask.shape[1] // 2
        arrow_length = min(cropped_mask.shape) // 3
        
        # Draw arrow using PC1 direction
        if rot_result['pc1'] is not None:
            pc1 = rot_result['pc1']
            dx = arrow_length * pc1[0]
            dy = arrow_length * pc1[1]
            
            color = 'green' if rot_result['is_reference'] else 'red'
            ax.arrow(center_x, center_y, dx, dy, 
                    color=color, width=3, head_width=15, head_length=10)
        
        title = f"{seg['label']} (ID: {seg['id']})\n"
        if rot_result['is_reference']:
            title += "REFERENCE: 0°"
        else:
            title += f"Angle: {rot_result['angle_to_reference']:.1f}°"
        title += f"\n(conf: {rot_result['confidence']:.2f})"
        
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    
    for idx in range(len(segments), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('relative_rotations_dotproduct.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
IMAGE_PATH = "input/room_test.png"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Step 1: Segment
segments, img_w, img_h, img_np = segment_image_simple(IMAGE_PATH, DEVICE)

# Step 2: Compute relative rotations (NEW FUNCTION NAME)
rotation_results, reference_id = estimate_relative_rotations_correct(segments)

# Step 3: Visualize
visualize_relative_rotations(segments, rotation_results, img_np, reference_id)

# Step 4: Print relationships
print("\n" + "="*80)
print("SPATIAL RELATIONSHIPS")
print("="*80)

for r in rotation_results:
    if r['is_reference']:
        continue
    
    angle = r['angle_to_reference']
    
    if angle < 15:
        relationship = "PARALLEL"
    elif 75 < angle < 105:
        relationship = "PERPENDICULAR"
    else:
        relationship = f"ANGLED ({angle:.1f}°)"
    
    print(f"{r['label']:20s} is {relationship:20s} to reference")